<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Collaborative_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaborative Filtering

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [ ]:
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
user_item_sparse = load_npz(
    "/content/drive/MyDrive/Artifact/user_item_sparse.npz"
)

user_encoder = joblib.load(
    "/content/drive/MyDrive/Artifact/user_encoder.pkl"
)

movie_encoder = joblib.load(
    "/content/drive/MyDrive/Artifact/movie_encoder.pkl"
)

movie_data = joblib.load(
    "/content/drive/MyDrive/Artifact/movie_data.pkl"
)

In [ ]:
print(user_item_sparse.shape)

(200948, 84432)


In [ ]:
svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

user_features = svd.fit_transform(user_item_sparse)

movie_features = svd.components_.T

In [ ]:
print(
    f"Explained Variance: "
    f"{svd.explained_variance_ratio_.sum():.4f}"
)

Explained Variance: 0.4091


In [ ]:
joblib.dump(
    svd,
    "/content/drive/MyDrive/Artifact/svd_model.pkl"
)

joblib.dump(
    user_features,
    "/content/drive/MyDrive/Artifact/user_features.pkl"
)

joblib.dump(
    movie_features,
    "/content/drive/MyDrive/Artifact/movie_features.pkl"
)

['/content/drive/MyDrive/Artifact/movie_features.pkl']

In [ ]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/ml-32m/ml-32m/ratings.csv"
)

In [17]:
def generate_candidates(
    user_id,
    candidate_size=500
):

    if user_id not in user_encoder.classes_:
        raise ValueError("Unknown User ID")

    user_index = user_encoder.transform(
        [user_id]
    )[0]

    user_vector = user_features[user_index]

    scores = movie_features @ user_vector

    top_candidates = np.argsort(
        scores
    )[::-1][:candidate_size]

    return top_candidates, scores

In [18]:
def filter_seen_movies(
    user_id,
    candidate_indices,
    scores
):

    watched = ratings.loc[
        ratings.userId == user_id,
        "movieId"
    ]

    watched = watched[
        watched.isin(
            movie_encoder.classes_
        )
    ]

    watched_indices = set(
        movie_encoder.transform(
            watched
        )
    )

    filtered = [
        idx
        for idx in candidate_indices
        if idx not in watched_indices
    ]

    return filtered

In [19]:
def rank_candidates(
    candidate_indices,
    scores,
    top_n=10
):

    top_indices = candidate_indices[:top_n]

    movie_ids = movie_encoder.inverse_transform(
        top_indices
    )

    recommendations = (
        movie_data[
            movie_data.movieId.isin(
                movie_ids
            )
        ]
        .copy()
    )

    score_map = {
        movie_encoder.inverse_transform([idx])[0]:
        scores[idx]
        for idx in top_indices
    }

    recommendations["svd_score"] = (
        recommendations.movieId
        .map(score_map)
    )

    return recommendations.sort_values(
        "svd_score",
        ascending=False
    )

In [26]:
def recommend_for_user(
    user_id,
    top_n=10
):

    candidates, scores = generate_candidates(
        user_id
    )

    filtered_candidates = filter_seen_movies(
        user_id,
        candidates,
        scores
    )

    recommendations = rank_candidates(
        filtered_candidates,
        scores,
        top_n
    )

    recommendations = (
        recommendations
        .sort_values("svd_score", ascending=False)
        .reset_index(drop=True)
    )

    recommendations.index.name = "Rank"

    return recommendations

In [27]:
recommend_for_user(
    user_id=1,
    top_n=10
)

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,858,"Godfather, The (1972)",Crime|Drama,1972.0,"Godfather, The","[Crime, Drama]",68646,238.0,al pacino atmospheric great acting masterpiece...,"Godfather, The Crime Drama Al Pacino atmospher...",2.863783
1,1222,Full Metal Jacket (1987),Drama|War,1987.0,Full Metal Jacket,"[Drama, War]",93058,600.0,vietnam war tumey s dvds imdb top 250 stanley ...,Full Metal Jacket Drama War Vietnam War Tumey'...,2.771993
2,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,1968.0,2001: A Space Odyssey,"[Adventure, Drama, Sci-Fi]",62622,62.0,atmospheric space stanley kubrick ambiguous at...,2001: A Space Odyssey Adventure Drama Sci-Fi a...,2.545833
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,1964.0,Dr. Strangelove or: How I Learned to Stop Worr...,"[Comedy, War]",57012,935.0,black and white stanley kubrick classic dark c...,Dr. Strangelove or: How I Learned to Stop Worr...,2.479119
4,2858,American Beauty (1999),Drama|Romance,1999.0,American Beauty,"[Drama, Romance]",169547,14.0,bittersweet great acting kevin spacey loneline...,American Beauty Drama Romance bittersweet grea...,2.291048
5,1250,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,1957.0,"Bridge on the River Kwai, The","[Adventure, Drama, War]",50212,826.0,tumey s dvds classic oscar best picture world ...,"Bridge on the River Kwai, The Adventure Drama ...",1.958763
6,1230,Annie Hall (1977),Comedy|Romance,1977.0,Annie Hall,"[Comedy, Romance]",75686,703.0,tumey s dvds woody allen funny overrated sarca...,Annie Hall Comedy Romance Tumey's DVDs Woody A...,1.841476
7,1307,When Harry Met Sally... (1989),Comedy|Romance,1989.0,When Harry Met Sally...,"[Comedy, Romance]",98635,639.0,romantic comedy best friends fall in love bill...,When Harry Met Sally... Comedy Romance romanti...,1.717384
8,1207,To Kill a Mockingbird (1962),Drama,1962.0,To Kill a Mockingbird,[Drama],56592,595.0,tumey s dvds book was better classic harper le...,To Kill a Mockingbird Drama Tumey's DVDs book ...,1.715159


In [28]:
recommend_for_user(10)

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,60074,Hancock (2008),Action|Adventure|Comedy|Crime|Fantasy,2008.0,Hancock,"[Action, Adventure, Comedy, Crime, Fantasy]",448157,8960.0,superhero will smith anti hero superhero point...,Hancock Action Adventure Comedy Crime Fantasy ...,2.551037
1,54286,"Bourne Ultimatum, The (2007)",Action|Crime|Thriller,2007.0,"Bourne Ultimatum, The","[Action, Crime, Thriller]",440963,2503.0,action adventure government research matt damo...,"Bourne Ultimatum, The Action Crime Thriller ac...",2.505721
2,145,Bad Boys (1995),Action|Comedy|Crime|Drama|Thriller,1995.0,Bad Boys,"[Action, Comedy, Crime, Drama, Thriller]",112442,9737.0,will smith action comedy funny martin lawrence...,Bad Boys Action Comedy Crime Drama Thriller Wi...,2.348487
3,5574,"Transporter, The (2002)",Action|Crime,2002.0,"Transporter, The","[Action, Crime]",293662,4108.0,action heist jason statham male nudity action ...,"Transporter, The Action Crime action heist Jas...",2.244943
4,30793,Charlie and the Chocolate Factory (2005),Adventure|Children|Comedy|Fantasy|IMAX,2005.0,Charlie and the Chocolate Factory,"[Adventure, Children, Comedy, Fantasy, IMAX]",367594,118.0,bad remake beautiful pictures helena bonham ca...,Charlie and the Chocolate Factory Adventure Ch...,2.195814
5,122892,Avengers: Age of Ultron (2015),Action|Adventure|Sci-Fi,2015.0,Avengers: Age of Ultron,"[Action, Adventure, Sci-Fi]",2395427,99861.0,joss whedon brian tyler danny elfman artificia...,Avengers: Age of Ultron Action Adventure Sci-F...,2.062170
6,1645,The Devil's Advocate (1997),Drama|Mystery|Thriller,1997.0,The Devil's Advocate,"[Drama, Mystery, Thriller]",118971,1813.0,al pacino deal with the devil keanu reeves rel...,The Devil's Advocate Drama Mystery Thriller Al...,2.046478
7,6541,"League of Extraordinary Gentlemen, The (a.k.a....",Action|Fantasy|Sci-Fi,2003.0,"League of Extraordinary Gentlemen, The (a.k.a....","[Action, Fantasy, Sci-Fi]",311429,8698.0,vampires gothic sean connery steampunk gothic ...,"League of Extraordinary Gentlemen, The (a.k.a....",1.996489
8,69844,Harry Potter and the Half-Blood Prince (2009),Adventure|Fantasy|Mystery|Romance|IMAX,2009.0,Harry Potter and the Half-Blood Prince,"[Adventure, Fantasy, Mystery, Romance, IMAX]",417741,767.0,daniel radcliffe disappointing harry potter ma...,Harry Potter and the Half-Blood Prince Adventu...,1.994957


In [29]:
recommend_for_user(25)

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,1036,Die Hard (1988),Action|Crime|Thriller,1988.0,Die Hard,"[Action, Crime, Thriller]",95016,562.0,bruce willis tumey s dvds must see always watc...,Die Hard Action Crime Thriller Bruce Willis Tu...,2.449486
1,1214,Alien (1979),Horror|Sci-Fi,1979.0,Alien,"[Horror, Sci-Fi]",78748,348.0,atmospheric sci fi thriller alien characters f...,Alien Horror Sci-Fi atmospheric sci-fi thrille...,2.037601
2,1527,"Fifth Element, The (1997)",Action|Adventure|Comedy|Sci-Fi,1997.0,"Fifth Element, The","[Action, Adventure, Comedy, Sci-Fi]",119116,18.0,chris tucker cinematography dystopic future sc...,"Fifth Element, The Action Adventure Comedy Sci...",1.878748
3,1374,Star Trek II: The Wrath of Khan (1982),Action|Adventure|Sci-Fi|Thriller,1982.0,Star Trek II: The Wrath of Khan,"[Action, Adventure, Sci-Fi, Thriller]",84726,154.0,james horner sci fi sci fi space space opera s...,Star Trek II: The Wrath of Khan Action Adventu...,1.831112
4,2028,Saving Private Ryan (1998),Action|Drama|War,1998.0,Saving Private Ryan,"[Action, Drama, War]",120815,857.0,steven spielberg cinematography historical his...,Saving Private Ryan Action Drama War Steven Sp...,1.549211
5,329,Star Trek: Generations (1994),Adventure|Drama|Sci-Fi,1994.0,Star Trek: Generations,"[Adventure, Drama, Sci-Fi]",111280,193.0,brent spiner guinan jonathan frakes patrick st...,Star Trek: Generations Adventure Drama Sci-Fi ...,1.529093
6,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,1980.0,Star Wars: Episode V - The Empire Strikes Back,"[Action, Adventure, Sci-Fi]",80684,1891.0,classic space tumey s dvds billy dee williams ...,Star Wars: Episode V - The Empire Strikes Back...,1.524932
7,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,1991.0,Star Trek VI: The Undiscovered Country,"[Action, Mystery, Sci-Fi]",102975,174.0,cliff eidelman star trek aliens sci fi space s...,Star Trek VI: The Undiscovered Country Action ...,1.444897
8,608,Fargo (1996),Comedy|Crime|Drama|Thriller,1996.0,Fargo,"[Comedy, Crime, Drama, Thriller]",116282,275.0,black comedy crime crime gone awry dark comedy...,Fargo Comedy Crime Drama Thriller black comedy...,1.396037
